# Online Retail Revenue Intelligence\n\nEnd-to-end analysis using Python, Pandas and Matplotlib. Source: [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail). The original Excel file should be downloaded from UCI and uploaded to the notebook environment.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nfrom pathlib import Path

## 1. Load and inspect data

In [ ]:
DATA_PATH = Path('/content/Online Retail.xlsx')  # update this path if needed\ndf = pd.read_excel(DATA_PATH)\nprint('Shape:', df.shape)\ndisplay(df.head())\nprint(df.dtypes)\ndisplay(df.isna().sum().sort_values(ascending=False))

## 2. Assess data quality

In [ ]:
print('Exact duplicate rows:', df.duplicated().sum())\nprint('Negative quantities:', (df['Quantity'] < 0).sum())\nprint('Zero quantities:', (df['Quantity'] == 0).sum())\nprint('Zero unit prices:', (df['UnitPrice'] == 0).sum())\nprint('Negative unit prices:', (df['UnitPrice'] < 0).sum())\nprint('Cancellation invoice rows:', df['InvoiceNo'].astype(str).str.startswith('C').sum())\ndisplay(df.loc[df['UnitPrice'] < 0])

## 3. Clean analytical copy and define eligible sales\n\nThe raw source is preserved. Exact duplicates are removed in the analytical copy. Negative quantities, cancellations, zero-price lines and negative-price accounting adjustments are retained for diagnostic work but excluded from the standard eligible-sales KPI definition.

In [ ]:
df_raw = df.copy()\ndf_clean = df_raw.drop_duplicates().copy()\ndf_clean['Description'] = df_clean['Description'].astype('string').str.strip()\ndf_clean['Country'] = df_clean['Country'].astype('string').str.strip()\ndf_clean['is_cancellation_invoice'] = df_clean['InvoiceNo'].astype(str).str.startswith('C')\ndf_clean['is_negative_quantity'] = df_clean['Quantity'] < 0\ndf_clean['is_zero_price'] = df_clean['UnitPrice'] == 0\ndf_clean['is_negative_price'] = df_clean['UnitPrice'] < 0\ndf_clean['LineRevenue'] = df_clean['Quantity'] * df_clean['UnitPrice']\nsales_mask = (df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0) & (~df_clean['is_cancellation_invoice'])\nsales_df = df_clean.loc[sales_mask].copy()\nprint('Rows after exact deduplication:', len(df_clean))\nprint('Eligible sales rows:', len(sales_df))\nprint('Gross sales revenue GBP:', round(sales_df['LineRevenue'].sum(), 2))\nprint('Eligible invoices:', sales_df['InvoiceNo'].nunique())\nprint('Identified customers:', sales_df['CustomerID'].nunique())\nprint('Units sold:', sales_df['Quantity'].sum())

## 4. Monthly trend

In [ ]:
sales_df['InvoiceDate'] = pd.to_datetime(sales_df['InvoiceDate'])\nsales_df['YearMonth'] = sales_df['InvoiceDate'].dt.to_period('M').astype(str)\nmonthly_sales = sales_df.groupby('YearMonth').agg(Revenue=('LineRevenue','sum'), Orders=('InvoiceNo','nunique'), UnitsSold=('Quantity','sum'), UniqueCustomers=('CustomerID','nunique')).reset_index().sort_values('YearMonth')\nmonthly_sales['AverageOrderValue'] = monthly_sales['Revenue'] / monthly_sales['Orders']\ndisplay(monthly_sales.round(2))\nplt.figure(figsize=(12,5))\nplt.plot(monthly_sales['YearMonth'], monthly_sales['Revenue'], marker='o')\nplt.title('Monthly Gross Sales Revenue')\nplt.xlabel('Month'); plt.ylabel('Revenue (£)'); plt.xticks(rotation=45); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

## 5. Product and country performance

In [ ]:
top_products = (sales_df.groupby(['StockCode','Description']).agg(Revenue=('LineRevenue','sum'), UnitsSold=('Quantity','sum'), Orders=('InvoiceNo','nunique')).reset_index().sort_values('Revenue', ascending=False))\ndisplay(top_products.head(10).round(2))\ncountry_sales = sales_df.groupby('Country').agg(Revenue=('LineRevenue','sum'), Orders=('InvoiceNo','nunique')).reset_index().sort_values('Revenue', ascending=False)\ndisplay(country_sales.head(10).round(2))

## 6. Customer repeat-purchase analysis

In [ ]:
customer_sales = sales_df.dropna(subset=['CustomerID']).copy()\ncustomer_metrics = customer_sales.groupby('CustomerID').agg(TotalRevenue=('LineRevenue','sum'), TotalOrders=('InvoiceNo','nunique'), TotalUnits=('Quantity','sum'), FirstPurchase=('InvoiceDate','min'), LastPurchase=('InvoiceDate','max')).reset_index()\ncustomer_metrics['CustomerType'] = customer_metrics['TotalOrders'].apply(lambda n: 'Repeat' if n > 1 else 'One-Time')\ncustomer_summary = customer_metrics.groupby('CustomerType').agg(Customers=('CustomerID','nunique'), Revenue=('TotalRevenue','sum'), Orders=('TotalOrders','sum'), UnitsSold=('TotalUnits','sum')).reset_index()\ncustomer_summary['RevenueSharePct'] = customer_summary['Revenue'] / customer_summary['Revenue'].sum() * 100\ndisplay(customer_summary.round(2))\nprint('Repeat purchase rate (%):', round((customer_metrics['TotalOrders'] > 1).mean() * 100, 2))\ndisplay(customer_metrics.sort_values('TotalRevenue', ascending=False).head(10).round(2))

## 7. Export analytical dataset\n\nThis CSV can be imported to SQL Server/Power BI. It may be large, so this repository documents the workflow rather than committing the large file directly.

In [ ]:
retail_analytics = df_clean.copy()\nretail_analytics['LineRevenue'] = retail_analytics['Quantity'] * retail_analytics['UnitPrice']\nretail_analytics['TransactionType'] = 'Sale'\nretail_analytics.loc[retail_analytics['is_cancellation_invoice'], 'TransactionType'] = 'Cancellation'\nretail_analytics.loc[retail_analytics['is_negative_quantity'] & (~retail_analytics['is_cancellation_invoice']), 'TransactionType'] = 'Negative Quantity - Review'\nretail_analytics.loc[retail_analytics['is_zero_price'], 'TransactionType'] = 'Zero Price - Review'\nretail_analytics.loc[retail_analytics['is_negative_price'], 'TransactionType'] = 'Negative Price - Accounting Review'\nretail_analytics.to_csv('online_retail_analytics.csv', index=False)\nprint('Export saved: online_retail_analytics.csv')

## Interpretation notes\n\n- December 2011 is partial: the dataset ends on 9 December.\n- A negative quantity indicates a negative-quantity transaction, not necessarily a confirmed customer return.\n- Gross sales revenue under the eligible-sales definition is not the same as reconciled net revenue or audited accounting revenue.\n- Bulk transactions and non-merchandise codes such as postage/manual charges should be investigated before product recommendations.